In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory, preload_memory
from google.genai import types
from dotenv import load_dotenv

print("✅ ADK components imported successfully.")

load_dotenv()

retry_config = types.HttpRetryOptions(
    attempts=3,  # Maximum retry attempts
    exp_base=5,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

**MemoryService vs. SessionService.** A `SessionService` (covered in `short_memory_management.ipynb`) holds the back-and-forth of a *single* conversation — ADK's short-term memory. A `MemoryService` is the long-term counterpart: a separate store you explicitly save session content into, then later search *across sessions* — even ones started fresh with a brand-new session id. `InMemoryMemoryService` is the dev/test implementation used here; production apps swap in a persistent backend (e.g. Vertex AI Memory Bank) without changing the rest of the code.

In [ ]:
async def run_session(
    runner_instance: Runner, user_queries: list[str] | str, session_id: str = "default"
):
    """Helper function to run queries in a session and display responses."""
    print(f"\n### Session: {session_id}")

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    # Convert single query to list
    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # Process each query
    for query in user_queries:
        print(f"\nUser > {query}")
        query_content = types.Content(role="user", parts=[types.Part(text=query)])

        # Stream agent response
        async for event in runner_instance.run_async(
            user_id=USER_ID, session_id=session.id, new_message=query_content
        ):
            if event.is_final_response() and event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"Model: > {text}")


print("✅ Helper functions defined.")

In [ ]:
memory_service = (
    InMemoryMemoryService()
)  # ADK's built-in Memory Service for development and testing

In [ ]:
# Define constants used throughout the notebook
APP_NAME = "MemoryDemoApp"
USER_ID = "demo_user"

# Create agent
user_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words.",
)

print("✅ Agent created")

In [ ]:
# Create Session Service
session_service = InMemorySessionService()  # Handles conversations

# Create runner with BOTH services
runner = Runner(
    agent=user_agent,
    app_name="MemoryDemoApp",
    session_service=session_service,
    memory_service=memory_service,  # Memory service is now available!
)

print("✅ Agent and Runner created with memory support!")

In [ ]:
# User tells agent about their favorite color
await run_session(
    runner,
    "My favorite color is blue-green. Can you write a Haiku about it?",
    "conversation-01",  # Session ID
)

In [ ]:
#see what is inside session
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="conversation-01"
)

# Let's see what's in the session
print("📝 Session contains:")
for event in session.events:
    text = (
        event.content.parts[0].text[:20]
        if event.content and event.content.parts
        else "(empty)"
    )
    print(f"  {event.content.role}: {text}...")

**Saving isn't automatic.** Having a `memory_service` wired into the `Runner` doesn't mean every session gets remembered — nothing is searchable across sessions until you explicitly call `memory_service.add_session_to_memory(session)`, as the next cell does. Skip that call and the conversation only ever lived in the (still in-memory) session object.

In [ ]:
#save session to the memory for long term memroy

await memory_service.add_session_to_memory(session)

print("Session added to memory")

## save memory - retrieve what we saved!

In [ ]:
# Create agent
user_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words. Use load_memory tool if you need to recall past conversations.",
    tools=[
        load_memory
    ],  # Agent now has access to Memory and can search it whenever it decides to!
)

print("✅ Agent with load_memory tool created.")



In [ ]:
# Create a new runner with the updated agent
runner = Runner(
    agent=user_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)



In [ ]:
await run_session(runner, "What is my favorite color?", "color-test")

In [ ]:
await run_session(runner, "My birthday is on March 15th.", "birthday-session-01")

In [ ]:
# Manually save the session to memory
birthday_session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="birthday-session-01"
)

await memory_service.add_session_to_memory(birthday_session)

print("✅ Birthday session saved to memory!")

In [ ]:
# Test retrieval in a NEW session
await run_session(
    runner, "When is my birthday?", "birthday-session-02"  # Different session ID
)

now you can see that by adding load_memory to the tools, now we can have access to the this memory between different sessions.

## search memory 

In [ ]:
# Search for color preferences
search_response = await memory_service.search_memory(
    app_name=APP_NAME, user_id=USER_ID, query="when is my favourite color?"
)

print("🔍 Search Results:")
print(f"  Found {len(search_response.memories)} relevant memories")
print()

for memory in search_response.memories:
    if memory.content and memory.content.parts:
        text = memory.content.parts[0].text[:80]
        print(f"  [{memory.author}]: {text}...")

**How `search_memory` works.** `search_memory(app_name, user_id, query)` looks across *all* sessions previously saved for that user/app and returns a `SearchMemoryResponse` with a `memories` list — each entry keeps the original `author` (`user` or the agent name) and `content`, much like a session event. `InMemoryMemoryService` does simple keyword matching under the hood; hosted backends (e.g. Vertex AI Memory Bank) do semantic/embedding-based retrieval instead, behind the same interface.

## autosaving

In [ ]:
async def auto_save_to_memory(callback_context):
    """Automatically save session to memory after each agent turn."""
    await callback_context._invocation_context.memory_service.add_session_to_memory(
        callback_context._invocation_context.session
    )


print("✅ Callback created.")


**What makes this "auto."** `after_agent_callback` is an ADK hook that fires automatically once the agent finishes producing its response for a turn — no manual call needed. Wiring `auto_save_to_memory` in there means every turn's session gets pushed into `memory_service` as a side effect, instead of relying on a developer to remember to call `add_session_to_memory` by hand.

In [ ]:
# Agent with automatic memory saving
auto_memory_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    name="AutoMemoryAgent",
    instruction="Answer user questions.",
    tools=[preload_memory],
    after_agent_callback=auto_save_to_memory,  # Saves after each turn!
)

print("✅ Agent created with automatic memory saving!")

**`load_memory` vs. `preload_memory`.** `load_memory` is a regular tool: the model itself decides mid-generation whether to call it, same as any other function tool. `preload_memory` isn't a decision the model makes — ADK runs a memory search *before* the agent starts generating and injects the results straight into context, so relevant memories are already there whether or not the model would have thought to ask.

In [ ]:
# Create a runner for the auto-save agent
# This connects our automated agent to the session and memory services
auto_runner = Runner(
    agent=auto_memory_agent,  # Use the agent with callback + preload_memory
    app_name=APP_NAME,
    session_service=session_service,  # Same services from Section 3
    memory_service=memory_service,
)

print("✅ Runner created.")

In [ ]:
# Test 1: Tell the agent about a gift (first conversation)
# The callback will automatically save this to memory when the turn completes
await run_session(
    auto_runner,
    "I gifted a new toy to my nephew on his 1st birthday!",
    "auto-save-test",
)

In [ ]:
# Test 2: Ask about the gift in a NEW session (second conversation)
# The agent should retrieve the memory using preload_memory and answer correctly
await run_session(
    auto_runner,
    "What did I gift my nephew?",
    "auto-save-test-2",  # Different session ID - proves memory works across sessions!
)